# Distant Contradiction Auditor - Legal-BERT Colab Training

Use a Colab runtime with **Runtime > Change runtime type > T4 GPU**. Run the cells from top to bottom.

The notebook fine-tunes `nlpaueb/legal-bert-base-uncased` and saves the trained model and evaluation metrics to Google Drive.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/dca')
DRIVE.mkdir(parents=True, exist_ok=True)
LOCAL = Path('/content')
print('Drive output directory:', DRIVE)

In [ ]:
import gzip
import shutil
from google.colab import files

def unzip_dataset(source, target):
    with gzip.open(source, 'rb') as source_file, target.open('wb') as target_file:
        shutil.copyfileobj(source_file, target_file)

candidates = [
    DRIVE / 'pairs_colab_full.jsonl',
    DRIVE / 'pairs_colab.jsonl',
    DRIVE / 'pairs.jsonl',
    LOCAL / 'pairs_colab_full.jsonl',
    LOCAL / 'pairs_colab.jsonl',
    LOCAL / 'pairs.jsonl',
]
compressed_candidates = [
    DRIVE / 'pairs_colab_full.jsonl.gz',
    DRIVE / 'pairs_colab.jsonl.gz',
    LOCAL / 'pairs_colab_full.jsonl.gz',
    LOCAL / 'pairs_colab.jsonl.gz',
]

PAIRS = next((path for path in candidates if path.exists()), None)
if PAIRS is None:
    compressed = next((path for path in compressed_candidates if path.exists()), None)
    if compressed is not None:
        PAIRS = compressed.with_suffix('')
        unzip_dataset(compressed, PAIRS)

if PAIRS is None:
    print('Upload pairs_colab.jsonl.gz or pairs_colab_full.jsonl.gz from the repository artifacts folder.')
    uploaded = files.upload()
    uploaded_path = LOCAL / next(iter(uploaded))
    PAIRS = uploaded_path
    if PAIRS.suffix == '.gz':
        uncompressed = PAIRS.with_suffix('')
        unzip_dataset(PAIRS, uncompressed)
        PAIRS = uncompressed

assert PAIRS.exists(), f'Dataset not found: {PAIRS}'
OUT = DRIVE / 'legalbert-dca'
print('Dataset:', PAIRS)
print('Output:', OUT)

In [ ]:
import json
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

MODEL = 'nlpaueb/legal-bert-base-uncased'
MAX_LEN = 256

rows = [json.loads(line) for line in PAIRS.read_text(encoding='utf-8', errors='replace').splitlines() if line.strip()]
by = {}
for row in rows:
    split_name = row.get('split', 'train')
    by.setdefault(split_name, []).append({
        'text_a': row['text_a'][:1200],
        'text_b': row['text_b'][:1200],
        'label': int(row['label']),
        'gap_bin': row.get('gap_bin', 'unknown'),
        'type': row.get('type', 'Inconsistency'),
    })
ds = DatasetDict({name: Dataset.from_list(values) for name, values in by.items()})
print(ds)

tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tokenizer(batch['text_a'], batch['text_b'], truncation=True, max_length=MAX_LEN, padding=False)

encoded = ds.map(tokenize, batched=True, remove_columns=['text_a', 'text_b'])
encoded = encoded.rename_column('label', 'labels')
print('Encoded splits:', encoded)

In [ ]:
import inspect
import torch
from sklearn.metrics import precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0
    )
    return {'precision': float(precision), 'recall': float(recall), 'f1': float(f1)}


eval_ds = encoded['dev'] if 'dev' in encoded else encoded['test']
training_parameters = inspect.signature(TrainingArguments.__init__).parameters
training_kwargs = {
    'output_dir': '/content/dca-checkpoints',
    'per_device_train_batch_size': 16,
    'per_device_eval_batch_size': 32,
    'gradient_accumulation_steps': 2,
    'num_train_epochs': 6,
    'learning_rate': 2e-5,
    'lr_scheduler_type': 'cosine',
    'weight_decay': 0.01,
    'fp16': torch.cuda.is_available(),
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1',
    'greater_is_better': True,
    'save_total_limit': 2,
    'logging_steps': 50,
    'report_to': [],
    'seed': 13,
}

if 'warmup_ratio' in training_parameters:
    training_kwargs['warmup_ratio'] = 0.1
else:
    steps_per_epoch = max(1, len(encoded['train']) // (16 * 2))
    training_kwargs['warmup_steps'] = max(1, int(steps_per_epoch * 6 * 0.1))

if 'eval_strategy' in training_parameters:
    training_kwargs['eval_strategy'] = 'epoch'
elif 'evaluation_strategy' in training_parameters:
    training_kwargs['evaluation_strategy'] = 'epoch'

args = TrainingArguments(**training_kwargs)
trainer_parameters = inspect.signature(Trainer.__init__).parameters
trainer_kwargs = {
    'model': model,
    'args': args,
    'train_dataset': encoded['train'],
    'eval_dataset': eval_ds,
    'compute_metrics': compute_metrics,
    'callbacks': [EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)],
}
if 'processing_class' in trainer_parameters:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**trainer_kwargs)

print('Starting training on', 'CUDA' if torch.cuda.is_available() else 'CPU')
trainer.train()
OUT.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)
print('Saved model to:', OUT)

In [ ]:
from sklearn.metrics import f1_score, precision_recall_fscore_support

test_result = trainer.predict(encoded['test'])
logits, labels = test_result.predictions, test_result.label_ids
predictions = np.argmax(logits, axis=-1)
precision, recall, f1, _ = precision_recall_fscore_support(
    labels, predictions, average='binary', zero_division=0
)
metrics = {
    'model': MODEL,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'test_precision': float(precision),
    'test_recall': float(recall),
    'test_f1': float(f1),
    'n_test': len(by['test']),
}
metrics_path = DRIVE / 'metrics_legalbert.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))
print('Metrics saved to:', metrics_path)

In [ ]:
import shutil

archive = shutil.make_archive('/content/legalbert-dca', 'zip', OUT)
print('Download this archive from the Colab file browser:', archive)
from google.colab import files
files.download(archive)